In [ ]:
import sys
sys.path.append('/data/users/quentin/package_paper/')
import scClone2DR as sccdr
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
import pickle
import torch
from tqdm import tqdm
import os
import pandas as pd
import plotly.io as pio
np.float_ = np.float64
rootpath = '/data/users/quentin/package_paper/experiments/paper_results'

COHORT = 'melanoma'
gene_set_collections = ['geneOncoKB','gene','c6','hallmarks', 'c2_pid']#, 'c2_kegg_medicus']
cohort2clonemodes = {'melanoma': ["scatrex",'scatrex_rawcounts_scvi', 'phenograph'], 'aml':['phenograph']}
gene_set_collection = gene_set_collections[3]
clonemode = cohort2clonemodes[COHORT][0]
mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)

directory = os.path.join(rootpath,COHORT)
if not os.path.exists(directory):
    os.makedirs(directory)
pathsave = os.path.join(rootpath,COHORT,gene_set_collection+'_'+clonemode)
# if not os.path.exists(pathsave):
#     os.makedirs(pathsave)
    
if COHORT=='melanoma':
    path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
    path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
    concentration_DMSO = '100'
    concentration_drug = '5'
    path_info_cohort = None
elif COHORT=='aml':
    concentration_DMSO = '200'
    concentration_drug = '10'
    path_rna = '/data/users/04_share_reanalysis_results/aml_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
    path_fastdrug = '/data/users/04_share_reanalysis_results/01_aml/AML_PCY_cell_numbers_no_plate_effect_correction.csv'
    path_info_cohort = '/data/users/04_share_reanalysis_results/01_aml/2024-08-15_aml_overview_scRNA.tsv'
model = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort, type_guide="lowrank_MVN", rank=10)
data_ref = model.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)



In [ ]:
rootpath = '/data/users/quentin/package_paper/experiments/paper_results'


In [ ]:
if False:
    idxs_train = [i for i in range(int(data_ref['N']))]
    idxs_test = []
    np.random.seed(10)
    data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, idxs_test)
    params_svi = model.train(data_train, penalty_l1=3.2, penalty_l2=1. , n_steps=2000)
    with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'wb') as handle:
        pickle.dump(params_svi, handle, protocol=pickle.HIGHEST_PROTOCOL)
        
    posterior_mean_params = model.sampling_from_posterior(data_ref, pathsave, params=params_svi, nb_ites=200, sample_names=model.sample_names)
    with open(os.path.join(rootpath,'{0}/{1}_{2}/posterior_mean_params.pkl'.format(COHORT, gene_set_collection, clonemode)), 'wb') as handle:
        pickle.dump(posterior_mean_params, handle, protocol=pickle.HIGHEST_PROTOCOL)
else:
    with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
        params_svi = pickle.load(handle)
        
    with open(os.path.join(rootpath,'{0}/{1}_{2}/posterior_mean_params.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
        res = pickle.load(handle)

In [ ]:
# mode of action
import pandas as pd
file_name = '/data/users/04_share_reanalysis_results/02_melanoma/MEL_drug_panels_fastDrug.xlsx'
dfaction = pd.read_excel(file_name)

In [ ]:
dfaction

In [ ]:
drug2action = {}
for drug in model.FD.selected_drugs:
    try:
        drug2action[drug] = dfaction[dfaction['Drug'].str.lower()==drug.lower()]['target_pathway'].iloc[0]
    except:
        print('Error for drug:', drug)
        print( dfaction[dfaction['Drug']==drug])

In [ ]:
unique_actions = np.unique(list(drug2action.values()))

In [ ]:
res = posterior_mean_params

In [ ]:
df = pd.read_csv(path_fastdrug)
df = df.loc[df['Concentration']==concentration_drug]
# Create a mapping from lowercased drug names to one of their original forms
drug_mapping = df.groupby(df['Drug'].str.lower())['Drug'].first().to_dict()
# Get the unique original names (one per lowercased drug)
selected_drugs = np.array([str(d) for d in drug_mapping.values() if (d!='DMSO' and not('|' in d))])
selected_drugs

In [ ]:
model.FD.selected_drugs = np.array(model.FD.selected_drugs)

In [ ]:
from matplotlib.colors import FuncNorm

def survival_probabilities_relative_by_patient_optimized(data, pi, props, cluster2clonelabel, cluster2cat, cat2clusters, selected_drugs, idxs2drugs=None, df_info_cohort=None, sampleID=0, savefig=None):
    """
    Here we normalize by the survival probability of the non malignant cells
    """
    from copy import deepcopy
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, Normalize
    from matplotlib.gridspec import GridSpec
    if idxs2drugs is None:
        idxs2drugs = range(data['D'])
    ratio_pi = deepcopy(pi)
    for idxdrug in range(pi.shape[0]):
        for sampleid in range(pi.shape[2]):
            learned_props = props[sampleid, :]
            healthy_survival = np.sum(
                np.nan_to_num(pi[idxdrug, :, sampleid] * learned_props)[cat2clusters['healthy']]
            )
            healthy_survival /= np.sum(learned_props[cat2clusters['healthy']])
            ratio_pi[idxdrug, :, sampleid] /= healthy_survival

    scores = np.sum(
        np.nan_to_num(ratio_pi)[:, cat2clusters['tumor'], :] * 
        (props[:, cat2clusters['tumor']].T)[None, :, :], axis=1
    )
    scores /= np.tile((np.sum(props[:, cat2clusters['tumor']], axis=1)).reshape(1, -1), (ratio_pi.shape[0], 1))
    scores = scores[:, sampleID]
    scores = np.clip(scores, a_min=0, a_max=10)

    for d in range(data['D']):
        pi[d, :, :][~(data['masks']['RNA'])] = float('nan')
        ratio_pi[d, :, :][~(data['masks']['RNA'])] = float('nan')

    available_clusters = np.array([
        i for i in range(pi.shape[1]) 
        if (not np.isnan(pi[0, i, sampleID]) and (cluster2cat[i] != 'healthy'))
    ]).astype(int)
    pi = pi[:, available_clusters, :]
    ratio_pi = ratio_pi[:, available_clusters, :]

    learned_props = props[sampleID, available_clusters]
    cluster2clonelabel = np.array(cluster2clonelabel)[available_clusters]
    cluster2cat = np.array(cluster2cat)[available_clusters]
    
    sorted_indices = np.argsort(learned_props)
    cluster2cat = cluster2cat[sorted_indices]
    learned_props = learned_props[sorted_indices]
    cluster2clonelabel = np.array(cluster2clonelabel)[sorted_indices]
    sorted_pi = pi[:, sorted_indices, sampleID]
    sorted_ratio_pi = ratio_pi[:, sorted_indices, sampleID]

    idxs_sort_drugs = np.argsort(scores)
    sorted_drugs = selected_drugs[idxs_sort_drugs]
    sorted_pi = sorted_pi[idxs_sort_drugs, :]
    sorted_ratio_pi = sorted_ratio_pi[idxs_sort_drugs, :]
    sorted_scores = scores[idxs_sort_drugs]
    
    past2new = {past:new for new, past in enumerate(idxs_sort_drugs)}
    idxs2drugs = np.sort([past2new[past] for past in idxs2drugs])

    label_colors = plt.colormaps['Oranges'](np.linspace(0, 1, 100))
    row_colors = np.array([label_colors[int(prop * 100)] for prop in learned_props])
    row_colors = row_colors.reshape(-1, 1, 4)

    fig = plt.figure(layout="constrained", figsize=(15, 8))
    gs = GridSpec(2, 2, figure=fig, height_ratios=[1, 0.1], width_ratios=[0.05, 1])

    ax1 = fig.add_subplot(gs[0, 0])
    # Replace ax1 with a horizontal bar chart
    ax1.clear()
    ax1.barh(
        y=np.arange(len(learned_props)), 
        width=learned_props, 
        color='orange', 
        edgecolor='black', 
        align='center', 
        height=0.9
    )
    ax1.set_yticks(np.arange(len(learned_props)))
    ax1.set_yticklabels([])
    ax1.invert_yaxis()  # Invert y-axis to match heatmap row order
    ax1.set_xlim(0, max(learned_props) * 1.1)  # Add padding to bar chart
    ax1.xaxis.set_label_position('top')
    ax1.xaxis.tick_top()
    ax1.set_xlabel("Clone size", fontsize=19, loc='left')
    #ax1.tick_params(axis='x', labelsize=13)
    ax1.set_xticks([])
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    for i, prop in enumerate(learned_props):
        ax1.text(
            prop * 1.02,  # Position the text slightly to the right of the bar
            i,  # Align with the bar
            f"{prop:.2f}",  # Display the value with 2 decimal places
            va='center',  # Vertically center the text
            fontsize=16,  # Font size for readability
            color='black'  # Text color
        )
    

    ax2 = fig.add_subplot(gs[0, 1])
    split = 0.7  # fraction of colorbar used for values < 1
    colors = [
        (0.0, (0, 0, 1)),     # blue
        (split, (1, 1, 1)),   # white at 1
        (1.0, (1, 0, 0))      # red
    ]    
    custom_cmap = LinearSegmentedColormap.from_list("terrain", colors)
    vmin = min([0.9, np.min(np.nan_to_num(sorted_ratio_pi[idxs2drugs,:]))])
    vmax = max([1.1, np.max(np.nan_to_num(sorted_ratio_pi[idxs2drugs,:]))])


    def forward(x):
        x = np.asarray(x)
        y = np.empty_like(x, dtype=float)

        below = x <= 1
        above = x > 1

        y[below] = split * (x[below] - vmin) / (1 - vmin)
        y[above] = split + (1 - split) * (x[above] - 1) / (vmax - 1)

        return y

    def inverse(y):
        y = np.asarray(y)
        x = np.empty_like(y, dtype=float)

        below = y <= split
        above = y > split

        x[below] = vmin + (y[below] / split) * (1 - vmin)
        x[above] = 1 + ((y[above] - split) / (1 - split)) * (vmax - 1)

        return x
    norm = FuncNorm((forward, inverse), vmin=vmin, vmax=vmax)
    sns.heatmap(sorted_ratio_pi.T, cmap=custom_cmap, norm=norm, cbar=True, alpha=0.7, ax=ax2, yticklabels=False)
    ax2.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    ax2.set_ylabel('Tumour clones', fontsize=17)
    colorbar = ax2.collections[0].colorbar
    colorbar.ax.tick_params(labelsize=15)

    
    # Example: more ticks below 1, fewer above
    ticks_below = np.linspace(vmin, 1, 8)      # dense
    ticks_above = np.linspace(1, vmax, 4)[1:]  # sparse, skip duplicate 1

    ticks = np.concatenate([ticks_below, ticks_above])

    colorbar.set_ticks(ticks)
    colorbar.set_ticklabels([f"{t:.2f}" for t in ticks])
    colorbar.set_label("Relative cluster\nsurvival probability", fontsize=17, labelpad=10)

    sorted_scores_filtered = sorted_scores[idxs2drugs]
    ax3 = fig.add_subplot(gs[1, 1], sharex=ax2)
    idxs = np.where(sorted_scores_filtered<=1)[0]
    ax3.plot(np.arange(len(sorted_scores_filtered))[idxs]+0.5, sorted_scores_filtered[idxs], marker='o', color='blue')
    idxs = np.where(sorted_scores_filtered>1)[0]
    ax3.plot(np.arange(len(sorted_scores_filtered))[idxs]+0.5, sorted_scores_filtered[idxs], marker='o', color='red')
    ax3.set_xticks([])
    ax3.set_xticks(np.arange(len(sorted_drugs[idxs2drugs]))+0.5)
    ax3.set_xticklabels(sorted_drugs[idxs2drugs], rotation=90, fontsize=15)
    ax3.set_ylabel("Relative tumor\nsurvival probability", fontsize=17)
    ax3.yaxis.set_tick_params(labelsize=17)

    ax3.grid(axis='y', linestyle='--', alpha=0.7)


    # plt.suptitle(f"Heatmap for sample: {sampleID}", ha='center', fontsize=18)

    if savefig:
        from pathlib import Path
        pathsavefig = f"/data/users/quentin/package_paper/experiments/figures/{COHORT}/figures_drug_response_{gene_set_collection}_{clonemode}/"
        Path(pathsavefig).mkdir(parents=True, exist_ok=True)

        plt.savefig(os.path.join(pathsavefig, savefig), dpi=250, bbox_inches='tight')
    plt.show()

In [ ]:
from copy import deepcopy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, FuncNorm
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches


def survival_probabilities_relative_by_patient_optimized(
    data, pi, props, cluster2clonelabel, cluster2cat, cat2clusters,
    selected_drugs,
    idxs2drugs=None, sampleID=0, savefig=None
):

    if idxs2drugs is None:
        idxs2drugs = range(data['D'])

    # ======================================================
    # Normalize by healthy survival
    # ======================================================
    ratio_pi = deepcopy(pi)

    for idxdrug in range(pi.shape[0]):
        for sampleid in range(pi.shape[2]):
            learned_props = props[sampleid, :]
            healthy_survival = np.sum(
                np.nan_to_num(pi[idxdrug, :, sampleid] * learned_props)[
                    cat2clusters['healthy']
                ]
            )
            healthy_survival /= np.sum(
                learned_props[cat2clusters['healthy']]
            )
            ratio_pi[idxdrug, :, sampleid] /= healthy_survival

    # ======================================================
    # Tumor survival scores
    # ======================================================
    scores = np.sum(
        np.nan_to_num(ratio_pi)[:, cat2clusters['tumor'], :]
        * (props[:, cat2clusters['tumor']].T)[None, :, :],
        axis=1
    )

    scores /= np.tile(
        np.sum(props[:, cat2clusters['tumor']], axis=1).reshape(1, -1),
        (ratio_pi.shape[0], 1)
    )

    scores = np.clip(scores[:, sampleID], 0, 10)

    # Mask invalid RNA
    for d in range(data['D']):
        pi[d, :, :][~(data['masks']['RNA'])] = np.nan
        ratio_pi[d, :, :][~(data['masks']['RNA'])] = np.nan

    # ======================================================
    # Remove healthy clusters
    # ======================================================
    available_clusters = np.array([
        i for i in range(pi.shape[1])
        if not np.isnan(pi[0, i, sampleID])
        and cluster2cat[i] != 'healthy'
    ]).astype(int)

    ratio_pi = ratio_pi[:, available_clusters, :]
    learned_props = props[sampleID, available_clusters]

    sorted_indices = np.argsort(learned_props)
    learned_props = learned_props[sorted_indices]
    sorted_ratio_pi = ratio_pi[:, sorted_indices, sampleID]

    # ======================================================
    # Sort drugs
    # ======================================================
    idxs_sort_drugs = np.argsort(scores)

    sorted_drugs = selected_drugs[idxs_sort_drugs]
    sorted_ratio_pi = sorted_ratio_pi[idxs_sort_drugs, :]
    sorted_scores = scores[idxs_sort_drugs]

    past2new = {past: new for new, past in enumerate(idxs_sort_drugs)}
    idxs2drugs = np.sort([past2new[p] for p in idxs2drugs])

    # Filtered values
    heatmap_data = sorted_ratio_pi[idxs2drugs, :]
    scores_plot = sorted_scores[idxs2drugs]
    drug_labels = sorted_drugs[idxs2drugs]

    n_drugs = len(drug_labels)

    # ======================================================
    # Figure Layout
    # ======================================================
    fig = plt.figure(figsize=(15, 9))

    gs = GridSpec(
        3, 4,
        figure=fig,
        height_ratios=[0.06, 1, 0.14],
        width_ratios=[0.1, 0.05, 1., 0.05],  # ← new spacer column
        hspace=0.1,
        wspace=0.1
    )

    # ======================================================
    # Clone size bar (left column)
    # ======================================================
    ax1 = fig.add_subplot(gs[1, 0])
    ax1.clear()

    ax1.barh(
        np.arange(len(learned_props))+0.5,
        learned_props,
        color="orange",
        edgecolor="black",
        height=0.9,
        left=0.1
    )
    ax1.set_yticks(np.arange(len(learned_props)))
    ax1.set_yticklabels([])
    ax1.invert_yaxis()  # Invert y-axis to match heatmap row order
    ax1.xaxis.set_label_position('top')
    ax1.xaxis.tick_top()
    ax1.set_xlabel("Clone size", fontsize=17, loc='left')
    #ax1.tick_params(axis='x', labelsize=13)
    ax1.set_xticks([])
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    for i, prop in enumerate(learned_props):
        ax1.text(
            prop + (i<3)* 0.12 - (i>=3)*0.085 ,  # Position the text slightly to the right of the bar
            i+0.5,  # Align with the bar
            f"{prop:.2f}",  # Display the value with 2 decimal places
            va='center',  # Vertically center the text
            fontsize=16,  # Font size for readability
            color='black'  # Text color
        )


    # ======================================================
    # Heatmap (center column)
    # ======================================================
    ax2 = fig.add_subplot(gs[1, 2])
    split = 0.7  # fraction of colorbar used for values < 1
    colors = [
        (0.0, (0, 0, 1)),     # blue
        (split, (1, 1, 1)),   # white at 1
        (1.0, (1, 0, 0))      # red
    ]    
    custom_cmap = LinearSegmentedColormap.from_list("terrain", colors)
    vmin = min([0.9, np.min(np.nan_to_num(sorted_ratio_pi[idxs2drugs,:]))])
    vmax = max([1.1, np.max(np.nan_to_num(sorted_ratio_pi[idxs2drugs,:]))])


    def forward(x):
        x = np.asarray(x)
        y = np.empty_like(x, dtype=float)

        below = x <= 1
        above = x > 1

        y[below] = split * (x[below] - vmin) / (1 - vmin)
        y[above] = split + (1 - split) * (x[above] - 1) / (vmax - 1)

        return y

    def inverse(y):
        y = np.asarray(y)
        x = np.empty_like(y, dtype=float)

        below = y <= split
        above = y > split

        x[below] = vmin + (y[below] / split) * (1 - vmin)
        x[above] = 1 + ((y[above] - split) / (1 - split)) * (vmax - 1)

        return x
    norm = FuncNorm((forward, inverse), vmin=vmin, vmax=vmax)
    

    hm = sns.heatmap(
        heatmap_data.T,
        cmap=custom_cmap,
        norm=norm,
        cbar=False,             # IMPORTANT
        ax=ax2,
        yticklabels=False
    )
    ax1.set_ylim(ax2.get_ylim())

    n_drugs = heatmap_data.shape[0]

    ax2.set_xlim(0, n_drugs)
    ax2.set_ylabel("Tumour clones", fontsize=17)
    ax2.tick_params(axis="x", bottom=False, labelbottom=False)

    # ======================================================
    # Dedicated colorbar axis (right column)
    # ======================================================
    cax = fig.add_subplot(gs[1, 3])
    
        

    colorbar = fig.colorbar(
        ax2.collections[0],
        cax=cax
    )
    colorbar.ax.tick_params(labelsize=15)
    colorbar.set_label("Relative cluster\nsurvival probability", fontsize=16, labelpad=10)


    # ======================================================
    # Mode-of-action bar (top center column)
    # ======================================================
    ax0 = fig.add_subplot(gs[0, 2], sharex=ax2)

    modes = [drug2action.get(d, "Unknown") for d in drug_labels]
    unique_modes = sorted(set(modes))
    colors = plt.cm.tab10.colors[:len(unique_modes)]
    unique_modes = sorted(unique_modes, key=len)
    mode2color = {m: colors[i] for i, m in enumerate(unique_modes)}

    for i, mode in enumerate(modes):
        ax0.bar(
            i + 0.5,
            1.,
            width=1,
            color=mode2color[mode]
        )

    ax0.set_ylim(0, 1)
    ax0.set_yticks([])
    ax0.tick_params(axis="x", bottom=False, labelbottom=False)
    ax0.spines[:].set_visible(False)


    # Optional legend
    patches = [
        mpatches.Patch(color=mode2color[m], label=m)
        for m in unique_modes
    ]


    ax0.legend(
        handles=patches,
        title="Mode of action",
        title_fontsize=16,
        bbox_to_anchor=(0.5, 4),
        loc="upper center",
        ncol=min(len(unique_modes), 5),
        frameon=False,
        fontsize=13
    )


    # ======================================================
    # Bottom survival plot (center column)
    # ======================================================
    ax3 = fig.add_subplot(gs[2, 2], sharex=ax2)

    low = np.where(scores_plot <= 1)[0]
    high = np.where(scores_plot > 1)[0]

    ax3.plot(low + 0.5, scores_plot[low], "o", color="blue")
    ax3.plot(high + 0.5, scores_plot[high], "o", color="red")

    ax3.set_xlim(0, n_drugs)
    ax3.set_xticks(np.arange(n_drugs) + 0.5)
    ax3.set_xticklabels(drug_labels, rotation=90, fontsize=15)

    ax3.set_ylabel("Relative tumor\nsurvival probability", fontsize=16)
    ax3.grid(axis="y", linestyle="--", alpha=0.6)
    ax3.yaxis.set_tick_params(labelsize=15)


    # ======================================================
    if savefig:
        plt.savefig(savefig, dpi=250, bbox_inches='tight')

    plt.show()

In [ ]:
for sampleID in range(len(model.sample_names)):
    print(sampleID)
    samplename = model.sample_names[sampleID]
    if COHORT=='aml':
        idx_dau = np.where(model.FD.selected_drugs=='Daunorubicin')[0][0]
        idxs2drugs = [i for i in range(len(model.FD.selected_drugs)) if i!=idx_dau]
        drugs = model.FD.selected_drugs
        pi = res['pi'].detach().numpy()
        try:
            survival_probabilities_relative_by_patient_optimized(data_ref, pi, params_svi['proportions'], model.cluster2clonelabel, model.cluster2cat, model.cat2clusters, drugs, idxs2drugs=idxs2drugs, sampleID=sampleID, savefig=f'{samplename}.png')
        except:
            print('sampleID error:', sampleID)
            pass
    else:
        survival_probabilities_relative_by_patient_optimized(data_ref, res['pi'].detach().numpy(), params_svi['proportions'], model.cluster2clonelabel, model.cluster2cat, model.cat2clusters, model.FD.selected_drugs, sampleID=sampleID, savefig=f'{samplename}.png')

In [ ]:
for i in [19, 21, 22, 46]:
    print(model.sample_names[i])

# Comparing survival probabilites across different gene sets

In [ ]:
with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
    params_svi = pickle.load(handle)
    props = params_svi['proportions']
    N = props.shape[0]
with open(os.path.join(rootpath,'{0}/{1}_{2}/posterior_mean_params.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
    res = pickle.load(handle)
    pi = res['pi'].detach().numpy().copy()
    D = pi.shape[0]
    Kmax = pi.shape[1]
pi = res['pi'].detach().numpy().copy()

pi_healthy = np.zeros((D, N))
norm = np.zeros(N)

for i in model.cat2clusters['healthy']:
    pi_healthy += props[:, i][None, :] * pi[:, i, :]
    norm += props[:, i]

norm = np.where(norm > 0, norm, np.nan)
pi_healthy /= norm[None, :]

pi /= pi_healthy[:, None, :]

In [ ]:
if COHORT=="melanoma":
    model.dfinfo = None

In [ ]:
import copy
#idxdrug = np.where(model.FD.selected_drugs=='Vosaroxin')[0][0]
for idxdrug, drug_name in enumerate(model.FD.selected_drugs):
    if drug_name=="Venetoclax":
        print(idxdrug)
        global_min = 1
        global_max = 1
        global_min = np.min([global_min, np.nanmin(pi[idxdrug,:,:])])
        global_max = np.max([global_max, np.nanmax(pi[idxdrug,:,:])])
        from pathlib import Path

        Path(f"/data/users/quentin/package_paper/experiments/figures/{COHORT}/survival_per_drug_{gene_set_collection}/").mkdir(parents=True, exist_ok=True)

        savefig = f"/data/users/quentin/package_paper/experiments/figures/{COHORT}/survival_per_drug_{gene_set_collection}/{drug_name}.png"
        survival_probabilities(data_ref, copy.deepcopy(pi), model.cluster2clonelabel, df_info_cohort=model.dfinfo, idxdrug=idxdrug, drug_name=None, savefig=savefig)
        survival_probabilities_venetoclax(data_ref, copy.deepcopy(pi), model.cluster2clonelabel,venetoclax_received_after, df_info_cohort=model.dfinfo, idxdrug=idxdrug, drug_name=None, savefig=savefig)
        survival_probabilities_venetoclax_mostresistant(data_ref, copy.deepcopy(pi), model.cluster2clonelabel,venetoclax_received_after, df_info_cohort=model.dfinfo, idxdrug=idxdrug, drug_name=None, savefig=savefig)

In [ ]:
for d in range(data_ref['D']):
    pi[d,:,:][~(data_ref['masks']['RNA'])] = float('nan')

In [ ]:
pi[70,idxs,:]

In [ ]:
model.sample_names

In [ ]:
df_aml = pd.read_csv('/data/users/quentin/final_package/experiments/survival_analysis/data/20240531_TPA_clinical_data_full_cohort.csv')
df_aml

In [ ]:
 df_aml[df_aml["SampleID"]==sample]['Last Treatment before TUPRO '].iloc[0]

In [ ]:
venetoclax_received_prev_lines = {}
venetoclax_received_after = {}
venetoclax_received_before = {}

for sample in model.sample_names:
    if df_aml[df_aml["SampleID"]==sample]['Venetoclax in previous lines '].iloc[0]=="yes":
        venetoclax_received_prev_lines[sample] = True
    else:
        venetoclax_received_prev_lines[sample] = False
        
    if "Venetoclax" in df_aml[df_aml["SampleID"]==sample]['Treatment after TUPRO'].iloc[0]:
        venetoclax_received_after[sample] = True
    else:
        venetoclax_received_after[sample] = False
        
    if "Venetoclax" in str(df_aml[df_aml["SampleID"]==sample]['Last Treatment before TUPRO '].iloc[0]):
        venetoclax_received_before[sample] = True
    else:
        venetoclax_received_before[sample] = False

        

    

In [ ]:
#pi[70,:,:]
idxs = np.where(np.array(model.cluster2clonelabel)=='tumor')[0]
np.sum(~np.isnan(pi[70,idxs,:]))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, Normalize
from matplotlib.colors import FuncNorm

def survival_probabilities(data, piorig, cluster2clonelabel, df_info_cohort=None, idxdrug=0, label_colors=None, drug_name=None,  savefig=None):
    # Comparing true survival probabilities and the one estimated
    label2name = {'healthy':'Non-malignant', 'tumor':'Cancer', 'putative':'Putative Cancer'}

    pi = piorig.copy()
    for d in range(data['D']):
        pi[d,:,:][~(data['masks']['RNA'])] = float('nan')

    # Sort clusters based on their labels
    sorted_indices = np.argsort(cluster2clonelabel)

    sorted_indices = np.arange(len(cluster2clonelabel))  #np.argsort(cluster2clonelabel)
    sorted_pi = pi[idxdrug, sorted_indices, :]

    if not(df_info_cohort is None):
        # Sort samples by patient_id and tissue_type
        sample_names = df_info_cohort.index.values
        sorted_meta_data = df_info_cohort.sort_values(by=['patient_id', 'tissue_type'])
        sorted_sample_ids = sorted_meta_data.index
        sample2idx = {sample: idx for idx, sample in enumerate(sorted_sample_ids.values)}
        sorted_indices_patient = np.array([sample2idx[sample] for sample in sample_names])
        sorted_pi = sorted_pi[:, sorted_indices_patient]

    # Get unique cluster labels and assign a color to each
    unique_labels = np.unique(cluster2clonelabel)
    if label_colors is None:
        label_colors = plt.colormaps['tab20'](np.linspace(0, 1, len(unique_labels)))

    # Create a color array for the y-axis labels, formatted for display
    row_colors = np.array([label_colors[np.where(unique_labels == label)[0][0]] for label in np.array(cluster2clonelabel)[sorted_indices]])

    # Reshape to a 2D array of RGB colors (n_clusters, 3) -> (n_clusters, 1, 3) for displaying as an image
    row_colors = row_colors.reshape(-1, 1, 4)  # Change 3 to 4 if the color has an alpha channel (RGBA)



    if not(df_info_cohort is None):
        # Create the main plot with two subplots: one for the color bar and one for the heatmap
        fig = plt.figure(layout="constrained")
        gs = GridSpec(2, 2, figure=fig, width_ratios=[0.05, 1], height_ratios=[1, 0.05])
        ax1 = fig.add_subplot(gs[0, 0])
        ax2 = fig.add_subplot(gs[0, 1])
    else:
        # Create the main plot with two subplots: one for the color bar and one for the heatmap
        fig, (ax1, ax2) = plt.subplots(ncols=2, gridspec_kw={"width_ratios": [0.005, 1]}, figsize=(12, 8))

    # Plot the color bar on the left using the label colors
    ax1.imshow(row_colors, aspect='auto', interpolation='nearest')
    ax1.axis('off')  # Remove the axis for the color bar

    # Plot the heatmap using seaborn with consistent vmin and vmax across all plots
    cax = fig.add_axes([1., 0.11, 0.02, 0.81])  # [left, bottom, width, height]
    
    split = 0.5
    colors = [
    (0.0, (0, 0, 1)),     # blue
    (split, (1, 1, 1)),   # white at 1
    (1.0, (1, 0, 0))      # red
    ]    
    custom_cmap = LinearSegmentedColormap.from_list("terrain", colors)
    import copy
    cmap = copy.copy(custom_cmap)
    cmap.set_bad('lightgray')
    cmap.set_bad('lightgray')
    vmin = min([0.9, np.min(np.nan_to_num(sorted_pi))])
    vmax = max([1.1, np.max(np.nan_to_num(sorted_pi))])
    def forward(x):
        x = np.asarray(x, dtype=float)

        y = np.empty_like(x)

        # Preserve NaNs
        nan_mask = np.isnan(x)

        below = (x <= 1) & (~nan_mask)
        above = (x > 1) & (~nan_mask)

        y[nan_mask] = np.nan
        y[below] = split * (x[below] - vmin) / (1 - vmin)
        y[above] = split + (1 - split) * (x[above] - 1) / (vmax - 1)

        return y

    def inverse(y):
        y = np.asarray(y)
        x = np.empty_like(y, dtype=float)

        below = y <= split
        above = y > split

        x[below] = vmin + (y[below] / split) * (1 - vmin)
        x[above] = 1 + ((y[above] - split) / (1 - split)) * (vmax - 1)

        return x
    
    norm = FuncNorm((forward, inverse), vmin=vmin, vmax=vmax)
    
    import numpy.ma as ma

    # Create masked array directly
    data_masked = ma.masked_invalid(sorted_pi)
    sns.heatmap(data_masked, cmap=cmap, norm=norm, cbar=True, ax=ax2, yticklabels=False, linewidths=0.5, linecolor='lightgray',  cbar_ax=cax, cbar_kws={"label": "Relative cluster\nsurvival probability"})
    # Get the colorbar and increase label fontsize
    cbar = ax2.collections[0].colorbar
    cbar.set_label("Relative cluster\nsurvival probability", fontsize=14)
    ax2.tick_params(axis='x', which='both', bottom=False, labelbottom=False)

    # Set the y-axis labels to the sorted cluster indices
    ax2.set_yticks(np.arange(len(sorted_indices)) + 0.5)

    # Custom legend for cluster labels
    label2clustername = {}
    for label in label2name.keys():
        if COHORT=='aml':
            label2clustername[label] = 'clusters'
        else:
            label2clustername[label] = 'clones'
    label2clustername['healthy'] = 'cells'
    handles = [mpatches.Patch(color=label_colors[i], label=f'{label2name[label]} {label2clustername[label]}') for i, label in enumerate(unique_labels)]
    if COHORT=='aml':
        ax2.legend(handles=handles, bbox_to_anchor=(0.5, 1.15), loc='upper center', fontsize=11, borderaxespad=0., ncols=2)
    else:
        ax2.legend(handles=handles, bbox_to_anchor=(0.5, 1.09), loc='upper center', borderaxespad=0., ncols=3)
    ax2.set_xticks([])

    # Optional: If you have patient labels
    # ax_heatmap.set_xticks(ticks=np.arange(sorted_pi.shape[1]))
    # ax_heatmap.set_xticklabels(patient_labels, rotation=90)
    if not(df_info_cohort is None):
        ax3 = fig.add_subplot(gs[1, 1])
        # Get unique cluster labels and assign a color to each
        unique_labels = np.unique(sorted_meta_data['patient_id'].values)
        label_colors = plt.colormaps['tab20'](np.linspace(0, 1, len(unique_labels)))
        # Create a color array for the y-axis labels, formatted for display
        col_colors = np.array([label_colors[np.where(unique_labels == label)[0][0]] for label in np.array(sorted_meta_data['patient_id'].values)])

        # Reshape to a 2D array of RGB colors (n_clusters, 3) -> (n_clusters, 1, 3) for displaying as an image
        col_colors = col_colors.reshape(1, -1, 4)  # Change 3 to 4 if the color has an alpha channel (RGBA)
        ax3.imshow(col_colors, aspect='auto', interpolation='nearest')
        #ax3.axis('off')  # Remove the axis for the color bar
        ax3.set_xlabel('Samples', fontsize=12)
        ax3.set_xticks([])
        ax3.set_yticks([])
    # Show the plot
    if not(drug_name is None):
        if drug_name == 'empty':
            pass
        else:
            plt.suptitle(f"Heatmap for the drug: {drug_name}")
            
    # Optional: Save the figure
    if savefig:
        plt.savefig(savefig, dpi=250, bbox_inches='tight')

    # Display the plot
    plt.show()



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, Normalize
from matplotlib.colors import FuncNorm

def survival_probabilities_venetoclax(data, piorig, cluster2clonelabel,venetoclax_received_dic, df_info_cohort=None, idxdrug=0, label_colors=None, drug_name=None,  savefig=None):
    # Comparing true survival probabilities and the one estimated
    label2name = {'healthy':'Non-malignant', 'tumor':'Cancer', 'putative':'Putative Cancer'}

    pi = piorig.copy()
    for d in range(data['D']):
        pi[d,:,:][~(data['masks']['RNA'])] = float('nan')

    # Sort clusters based on their labels
    sorted_indices = np.argsort(cluster2clonelabel)

    sorted_indices = np.arange(len(cluster2clonelabel))  #np.argsort(cluster2clonelabel)
    sorted_pi = pi[idxdrug, sorted_indices, :]

    venetoclax_received=[]
    if not(df_info_cohort is None):
        # Sort samples by patient_id and tissue_type
        sample_names = df_info_cohort.index.values
        sorted_meta_data = df_info_cohort.sort_values(by=['patient_id', 'tissue_type'])
        sorted_sample_ids = sorted_meta_data.index
        sample2idx = {sample: idx for idx, sample in enumerate(sorted_sample_ids.values)}
        sorted_indices_patient = np.array([sample2idx[sample] for sample in sample_names])
        venetoclax_received = [venetoclax_received_dic[sample] for sample in sorted_sample_ids]
        
        sorted_pi = sorted_pi[:, sorted_indices_patient]

    # Get unique cluster labels and assign a color to each
    unique_labels = np.unique(cluster2clonelabel)
    if label_colors is None:
        label_colors = plt.colormaps['tab20'](np.linspace(0, 1, len(unique_labels)))

    # Create a color array for the y-axis labels, formatted for display
    row_colors = np.array([label_colors[np.where(unique_labels == label)[0][0]] for label in np.array(cluster2clonelabel)[sorted_indices]])

    # Reshape to a 2D array of RGB colors (n_clusters, 3) -> (n_clusters, 1, 3) for displaying as an image
    row_colors = row_colors.reshape(-1, 1, 4)  # Change 3 to 4 if the color has an alpha channel (RGBA)



    if not(df_info_cohort is None):
        # Create the main plot with two subplots: one for the color bar and one for the heatmap
        fig = plt.figure(layout="constrained")
        gs = GridSpec(2, 2, figure=fig, width_ratios=[0.05, 1], height_ratios=[1, 0.05])
        ax1 = fig.add_subplot(gs[0, 0])
        ax2 = fig.add_subplot(gs[0, 1])
    else:
        # Create the main plot with two subplots: one for the color bar and one for the heatmap
        fig, (ax1, ax2) = plt.subplots(ncols=2, gridspec_kw={"width_ratios": [0.005, 1]}, figsize=(12, 8))

    # Plot the color bar on the left using the label colors
    ax1.imshow(row_colors, aspect='auto', interpolation='nearest')
    ax1.axis('off')  # Remove the axis for the color bar

    # Plot the heatmap using seaborn with consistent vmin and vmax across all plots
    cax = fig.add_axes([1., 0.11, 0.02, 0.81])  # [left, bottom, width, height]
    
    split = 0.5
    colors = [
    (0.0, (0, 0, 1)),     # blue
    (split, (1, 1, 1)),   # white at 1
    (1.0, (1, 0, 0))      # red
    ]    
    custom_cmap = LinearSegmentedColormap.from_list("terrain", colors)
    import copy
    cmap = copy.copy(custom_cmap)
    cmap.set_bad('lightgray')
    cmap.set_bad('lightgray')
    vmin = min([0.9, np.min(np.nan_to_num(sorted_pi))])
    vmax = max([1.1, np.max(np.nan_to_num(sorted_pi))])
    def forward(x):
        x = np.asarray(x, dtype=float)

        y = np.empty_like(x)

        # Preserve NaNs
        nan_mask = np.isnan(x)

        below = (x <= 1) & (~nan_mask)
        above = (x > 1) & (~nan_mask)

        y[nan_mask] = np.nan
        y[below] = split * (x[below] - vmin) / (1 - vmin)
        y[above] = split + (1 - split) * (x[above] - 1) / (vmax - 1)

        return y

    def inverse(y):
        y = np.asarray(y)
        x = np.empty_like(y, dtype=float)

        below = y <= split
        above = y > split

        x[below] = vmin + (y[below] / split) * (1 - vmin)
        x[above] = 1 + ((y[above] - split) / (1 - split)) * (vmax - 1)

        return x
    
    norm = FuncNorm((forward, inverse), vmin=vmin, vmax=vmax)
    
    import numpy.ma as ma

    # Create masked array directly
    data_masked = ma.masked_invalid(sorted_pi)
    sns.heatmap(data_masked, cmap=cmap, norm=norm, cbar=True, ax=ax2, yticklabels=False, linewidths=0.5, linecolor='lightgray',  cbar_ax=cax, cbar_kws={"label": "Relative cluster\nsurvival probability"})
    # Get the colorbar and increase label fontsize
    cbar = ax2.collections[0].colorbar
    cbar.set_label("Relative cluster\nsurvival probability", fontsize=14)
    ax2.tick_params(axis='x', which='both', bottom=False, labelbottom=False)

    # Set the y-axis labels to the sorted cluster indices
    ax2.set_yticks(np.arange(len(sorted_indices)) + 0.5)
    

    # Custom legend for cluster labels
    label2clustername = {}
    for label in label2name.keys():
        if COHORT=='aml':
            label2clustername[label] = 'clusters'
        else:
            label2clustername[label] = 'clones'
    label2clustername['healthy'] = 'cells'
    handles = [mpatches.Patch(color=label_colors[i], label=f'{label2name[label]} {label2clustername[label]}') for i, label in enumerate(unique_labels)]
    # Add dot legend entry
    from matplotlib.lines import Line2D

    handles.append(
        Line2D([0], [0],
               marker='o',
               color='w',
               markerfacecolor='black',
               markersize=6,
               label='Samples who received the drug')
    )
    if COHORT=='aml':
        ax2.legend(handles=handles, bbox_to_anchor=(0.5, 1.15), loc='upper center', fontsize=11, borderaxespad=0., ncols=2)
    else:
        ax2.legend(handles=handles, bbox_to_anchor=(0.5, 1.09), loc='upper center', borderaxespad=0., ncols=3)
    
    ax2.set_xticks([])

    # Optional: If you have patient labels
    # ax_heatmap.set_xticks(ticks=np.arange(sorted_pi.shape[1]))
    # ax_heatmap.set_xticklabels(patient_labels, rotation=90)
    if not(df_info_cohort is None):
        ax3 = fig.add_subplot(gs[1, 1])
        # Get unique cluster labels and assign a color to each
        unique_labels = np.unique(sorted_meta_data['patient_id'].values)
        label_colors = plt.colormaps['tab20'](np.linspace(0, 1, len(unique_labels)))
        # Create a color array for the y-axis labels, formatted for display
        col_colors = np.array([label_colors[np.where(unique_labels == label)[0][0]] for label in np.array(sorted_meta_data['patient_id'].values)])

        # Reshape to a 2D array of RGB colors (n_clusters, 3) -> (n_clusters, 1, 3) for displaying as an image
        col_colors = col_colors.reshape(1, -1, 4)  # Change 3 to 4 if the color has an alpha channel (RGBA)
        ax3.imshow(col_colors, aspect='auto', interpolation='nearest')
        #ax3.axis('off')  # Remove the axis for the color bar
        ax3.set_xlabel('Samples', fontsize=12)
        ax3.set_xticks([])
        ax3.set_yticks([])
        
        
        # Add dots for Venetoclax
        ven = venetoclax_received
        x_positions = np.where(ven)[0]

        ax3.scatter(
            x_positions,
            np.zeros_like(x_positions),
            color='black',
            s=15,
            zorder=3
        )
        
        
        
    # Show the plot
    if not(drug_name is None):
        if drug_name == 'empty':
            pass
        else:
            plt.suptitle(f"Heatmap for the drug: {drug_name}")
            
    # Optional: Save the figure
    if savefig:
        plt.savefig(savefig, dpi=250, bbox_inches='tight')

    # Display the plot
    plt.show()



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, Normalize
from matplotlib.colors import FuncNorm

def survival_probabilities_venetoclax_mostresistant(data, piorig, cluster2clonelabel,venetoclax_received_dic, df_info_cohort=None, idxdrug=0, label_colors=None, drug_name=None,  savefig=None):
    # Comparing true survival probabilities and the one estimated
    label2name = {'healthy':'Non-malignant', 'tumor':'Cancer', 'putative':'Putative Cancer'}

    pi = piorig.copy()
    for d in range(data['D']):
        pi[d,:,:][~(data['masks']['RNA'])] = float('nan')

    # Sort clusters based on their labels
    sorted_indices = np.argsort(cluster2clonelabel)

    sorted_indices = np.arange(len(cluster2clonelabel))  #np.argsort(cluster2clonelabel)
    sorted_pi = pi[idxdrug, sorted_indices, :]

    venetoclax_received=[]
    if not(df_info_cohort is None):
        # Sort samples by patient_id and tissue_type
        sample_names = df_info_cohort.index.values
        sorted_meta_data = df_info_cohort.sort_values(by=['patient_id', 'tissue_type'])
        sorted_sample_ids = sorted_meta_data.index
        sample2idx = {sample: idx for idx, sample in enumerate(sorted_sample_ids.values)}
        sorted_indices_patient = np.array([sample2idx[sample] for sample in sample_names])
        venetoclax_received = [venetoclax_received_dic[sample] for sample in sorted_sample_ids]
        
        sorted_pi = sorted_pi[:, sorted_indices_patient]

    # Get unique cluster labels and assign a color to each
    unique_labels = np.unique(cluster2clonelabel)
    if label_colors is None:
        label_colors = plt.colormaps['tab20'](np.linspace(0, 1, len(unique_labels)))

    # Create a color array for the y-axis labels, formatted for display
    row_colors = np.array([label_colors[np.where(unique_labels == label)[0][0]] for label in np.array(cluster2clonelabel)[sorted_indices]])

    # Reshape to a 2D array of RGB colors (n_clusters, 3) -> (n_clusters, 1, 3) for displaying as an image
    row_colors = row_colors.reshape(-1, 1, 4)  # Change 3 to 4 if the color has an alpha channel (RGBA)



    if not(df_info_cohort is None):
        # Create the main plot with two subplots: one for the color bar and one for the heatmap
        fig = plt.figure(layout="constrained")
        gs = GridSpec(3, 2, figure=fig, width_ratios=[0.05, 1], height_ratios=[1, 0.05, 0.05])
        ax1 = fig.add_subplot(gs[0, 0])
        ax2 = fig.add_subplot(gs[0, 1])
    else:
        # Create the main plot with two subplots: one for the color bar and one for the heatmap
        fig, (ax1, ax2) = plt.subplots(ncols=2, gridspec_kw={"width_ratios": [0.005, 1]}, figsize=(12, 8))

    # Plot the color bar on the left using the label colors
    ax1.imshow(row_colors, aspect='auto', interpolation='nearest')
    ax1.axis('off')  # Remove the axis for the color bar

    # Plot the heatmap using seaborn with consistent vmin and vmax across all plots
    cax = fig.add_axes([1., 0.11, 0.02, 0.81])  # [left, bottom, width, height]
    
    split = 0.5
    colors = [
    (0.0, (0, 0, 1)),     # blue
    (split, (1, 1, 1)),   # white at 1
    (1.0, (1, 0, 0))      # red
    ]    
    custom_cmap = LinearSegmentedColormap.from_list("terrain", colors)
    import copy
    cmap = copy.copy(custom_cmap)
    cmap.set_bad('lightgray')
    cmap.set_bad('lightgray')
    vmin = min([0.9, np.min(np.nan_to_num(sorted_pi))])
    vmax = max([1.1, np.max(np.nan_to_num(sorted_pi))])
    def forward(x):
        x = np.asarray(x, dtype=float)

        y = np.empty_like(x)

        # Preserve NaNs
        nan_mask = np.isnan(x)

        below = (x <= 1) & (~nan_mask)
        above = (x > 1) & (~nan_mask)

        y[nan_mask] = np.nan
        y[below] = split * (x[below] - vmin) / (1 - vmin)
        y[above] = split + (1 - split) * (x[above] - 1) / (vmax - 1)

        return y

    def inverse(y):
        y = np.asarray(y)
        x = np.empty_like(y, dtype=float)

        below = y <= split
        above = y > split

        x[below] = vmin + (y[below] / split) * (1 - vmin)
        x[above] = 1 + ((y[above] - split) / (1 - split)) * (vmax - 1)

        return x
    
    norm = FuncNorm((forward, inverse), vmin=vmin, vmax=vmax)
    
    import numpy.ma as ma

    # Create masked array directly
    data_masked = ma.masked_invalid(sorted_pi)
    sns.heatmap(data_masked, cmap=cmap, norm=norm, cbar=True, ax=ax2, yticklabels=False, linewidths=0.5, linecolor='lightgray',  cbar_ax=cax, cbar_kws={"label": "Relative cluster\nsurvival probability"})
    # Get the colorbar and increase label fontsize
    cbar = ax2.collections[0].colorbar
    cbar.set_label("Relative cluster\nsurvival probability", fontsize=14)
    ax2.tick_params(axis='x', which='both', bottom=False, labelbottom=False)

    # Set the y-axis labels to the sorted cluster indices
    ax2.set_yticks(np.arange(len(sorted_indices)) + 0.5)
    

    # Custom legend for cluster labels
    label2clustername = {}
    for label in label2name.keys():
        if COHORT=='aml':
            label2clustername[label] = 'clusters'
        else:
            label2clustername[label] = 'clones'
    label2clustername['healthy'] = 'cells'
    handles = [mpatches.Patch(color=label_colors[i], label=f'{label2name[label]} {label2clustername[label]}') for i, label in enumerate(unique_labels)]
    # Add dot legend entry
    from matplotlib.lines import Line2D

    handles.append(
        Line2D([0], [0],
               marker='o',
               color='w',
               markerfacecolor='black',
               markersize=6,
               label='Received the drug')
    )
    if COHORT=='aml':
        ax2.legend(handles=handles, bbox_to_anchor=(0.5, 1.15), loc='upper center', fontsize=11, borderaxespad=0., ncols=2)
    else:
        ax2.legend(handles=handles, bbox_to_anchor=(0.5, 1.09), loc='upper center', borderaxespad=0., ncols=3)
    
    ax2.set_xticks([])

    # Optional: If you have patient labels
    # ax_heatmap.set_xticks(ticks=np.arange(sorted_pi.shape[1]))
    # ax_heatmap.set_xticklabels(patient_labels, rotation=90)
    if not(df_info_cohort is None):
        ax3 = fig.add_subplot(gs[2, 1])
        # Get unique cluster labels and assign a color to each
        unique_labels = np.unique(sorted_meta_data['patient_id'].values)
        label_colors = plt.colormaps['tab20'](np.linspace(0, 1, len(unique_labels)))
        # Create a color array for the y-axis labels, formatted for display
        col_colors = np.array([label_colors[np.where(unique_labels == label)[0][0]] for label in np.array(sorted_meta_data['patient_id'].values)])

        # Reshape to a 2D array of RGB colors (n_clusters, 3) -> (n_clusters, 1, 3) for displaying as an image
        col_colors = col_colors.reshape(1, -1, 4)  # Change 3 to 4 if the color has an alpha channel (RGBA)
        ax3.imshow(col_colors, aspect='auto', interpolation='nearest')
        #ax3.axis('off')  # Remove the axis for the color bar
        ax3.set_xlabel('Samples', fontsize=12)
        ax3.set_xticks([])
        ax3.set_yticks([])
        
        
        # Add dots for Venetoclax
        ven = venetoclax_received
        x_positions = np.where(ven)[0]

        ax3.scatter(
            x_positions,
            np.zeros_like(x_positions),
            color='black',
            s=15,
            zorder=3
        )
        
        ax4 = fig.add_subplot(gs[1, 1])
        idxstumor = np.where(np.array(cluster2clonelabel)!="healthy")[0]
        # Identify columns that were entirely NaN
        pitumor = np.max(np.nan_to_num(sorted_pi[idxstumor, :]), axis=0)

        nan_cols = np.isnan(sorted_pi[idxstumor, :]).all(axis=0)

        # Map values to colors using the same cmap/norm as the heatmap
        pitumor_colors = cmap(norm(pitumor))

        # Set color to gray where all values were NaN
        pitumor_colors[nan_cols] = [0.5, 0.5, 0.5, 1.0]
        pitumor_colors = pitumor_colors.reshape(1, -1, 4)
        ax4.imshow(pitumor_colors, aspect='auto', interpolation='nearest')
        
        ax4.set_xlabel('Most resistant (putative) cancer cluster', fontsize=12)
        ax4.set_xticks([])
        ax4.set_yticks([])
        
        # Fisher test
        from scipy.stats import fisher_exact
        idxs_valid_samples = np.where(~nan_cols)[0]
        table = np.zeros((2,2))
        for i, idx_samp in enumerate(idxs_valid_samples):
            if pitumor[idx_samp]<1:
                if venetoclax_received[i]:
                    table[0,0] += 1
                else:
                    table[0,1] += 1
            else:
                if venetoclax_received[i]:
                    table[1,0] += 1
                else:
                    table[1,1] += 1
        print(table)
        
        res = fisher_exact(table, alternative='two-sided')
        print("p value Fisher exact test: ", res)
        
        
                         
                         
    # Show the plot
    if not(drug_name is None):
        if drug_name == 'empty':
            pass
        else:
            plt.suptitle(f"Heatmap for the drug: {drug_name}")
            
    # Optional: Save the figure
    if savefig:
        plt.savefig(savefig, dpi=250, bbox_inches='tight')

    # Display the plot
    plt.show()

